In [1]:
import numpy as np
import random

# ==========================================
# 1. DEFINE THE GRIDWORLD ENVIRONMENT
# ==========================================
GRID_SIZE = 4
START_STATE = (0, 0)
GOAL_STATE = (3, 3)

# Actions: 0 = Up, 1 = Down, 2 = Left, 3 = Right
ACTIONS = [0, 1, 2, 3]
NUM_ACTIONS = len(ACTIONS)
NUM_STATES = GRID_SIZE * GRID_SIZE

def state_to_index(state):
    """Converts a 2D coordinate (row, col) into a flat 1D array index."""
    return state[0] * GRID_SIZE + state[1]

def step(state, action):
    """Executes an action in the world, returning (next_state, reward, done)."""
    row, col = state
    
    if action == 0:   # Up
        row = max(0, row - 1)
    elif action == 1: # Down
        row = min(GRID_SIZE - 1, row + 1)
    elif action == 2: # Left
        col = max(0, col - 1)
    elif action == 3: # Right
        col = min(GRID_SIZE - 1, col + 1)
        
    next_state = (row, col)
    
    # Calculate feedback loop metrics
    if next_state == GOAL_STATE:
        return next_state, 10.0, True  # Big win reward!
    else:
        return next_state, -1.0, False # Time penalty penalty

# ==========================================
# 2. HYPERPARAMETERS & Q-TABLE INITIALIZATION
# ==========================================
alpha = 0.1    # Learning rate (how fast it accepts new info)
gamma = 0.9    # Discount factor (how much it values future rewards vs immediate ones)
epsilon = 1.0  # Exploration rate (starts at 100% random exploration)
decay = 0.995  # How fast we stop exploring randomly and start trusting the cheat sheet

# Rows = 16 unique tile positions, Columns = 4 directions
q_table = np.zeros((NUM_STATES, NUM_ACTIONS))

# ==========================================
# 3. THE TRAINING LOOP
# ==========================================
EPISODES = 500
print(f"Training agent over {EPISODES} practice runs...")

for episode in range(EPISODES):
    current_state = START_STATE
    done = False
    
    while not done:
        state_idx = state_to_index(current_state)
        
        # Epsilon-Greedy Action Selection Strategy
        if random.uniform(0, 1) < epsilon:
            action = random.choice(ACTIONS) # Explore: Pick a random direction
        else:
            action = np.argmax(q_table[state_idx]) # Exploit: Use the best known move
            
        # Move the agent and see what happens
        next_state, reward, done = step(current_state, action)
        next_state_idx = state_to_index(next_state)
        
        # The Bellman Equation Core Update Formula
        old_value = q_table[state_idx, action]
        next_max = np.max(q_table[next_state_idx])
        
        # Compute the updated estimation value
        new_value = old_value + alpha * (reward + gamma * next_max - old_value)
        q_table[state_idx, action] = new_value
        
        current_state = next_state
        
    # Gradually decrease exploration as the agent gathers experience
    epsilon *= decay

print("Training finished successfully!\n")

# ==========================================
# 4. TESTING THE TRAINED BRAIN
# ==========================================
print("--- Testing Learned Optimal Path ---")
test_state = START_STATE
path = [test_state]
steps = 0

while test_state != GOAL_STATE and steps < 20:
    s_idx = state_to_index(test_state)
    best_action = np.argmax(q_table[s_idx])
    
    # Map directional indices back to text arrows
    arrows = ["↑", "↓", "←", "→"]
    print(f"Position: {test_state} -> Chooses Action: {arrows[best_action]}")
    
    test_state, _, _ = step(test_state, best_action)
    path.append(test_state)
    steps += 1

print(f"Final Path Taken: {path}")

Training agent over 500 practice runs...
Training finished successfully!

--- Testing Learned Optimal Path ---
Position: (0, 0) -> Chooses Action: →
Position: (0, 1) -> Chooses Action: →
Position: (0, 2) -> Chooses Action: ↓
Position: (1, 2) -> Chooses Action: →
Position: (1, 3) -> Chooses Action: ↓
Position: (2, 3) -> Chooses Action: ↓
Final Path Taken: [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (2, 3), (3, 3)]
